# Session 12: Peak detection and feature extraction

A quality analyst at a supplier or a pharmacy decides whether a commercial lavender oil meets the European Pharmacopoeia's chromatographic profile, read as the area percent of each component in a gas chromatogram with flame ionization detection (GC-FID): camphor at most 1.2%, linalool 20.0 to 45.0% and linalyl acetate 25.0 to 47.0%. An area percent depends on which peaks count and where each starts and ends. This notebook finds the peaks in a run with `find_peaks`, with settings chosen from the noise and the smallest peak that counts, integrates each between bounds set by its neighbors, and reports the three components. Section 1 works the method on Dataset A. Section 2 applies it to the three replicate runs of your own oil, by hand and by a skill you write for Claude Code and check against the hand-built table. Section 3 compares two integration rules and asks whether your call depends on the choice. Section 4 is the judgment cell.

The data are real: GC-FID runs of commercial lavender oils on a polar (wax) column, from K. Pokajewicz (2024), "Lavandula angustifolia essential oil adulteration dataset", Mendeley Data, version 1, [doi:10.17632/pgkrc7wyj4.1](https://doi.org/10.17632/pgkrc7wyj4.1), CC BY 4.0, accessed 2026-10-22. `data/README.md` describes the files, which oil each was renamed from, and what each holds.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 12 assignment on Canvas by 1:30 pm on Thursday, November 12. Submissions are individual. Every student's runs are in `data/`, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart and holds your prompt record from section 2, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys`, `Path` from `pathlib`, NumPy, pandas and Matplotlib, and four new functions. `netcdf_file` from `scipy.io` reads a run saved by the data system in the AIA netCDF format, the chromatography exchange format. `find_peaks` from `scipy.signal` finds the local maxima of a signal that meet the settings you give it, and `peak_widths` measures each one's width at half its height. `trapezoid` from `scipy.integrate` adds up the area under a curve. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from scipy.io import netcdf_file
from scipy.signal import find_peaks, peak_widths

## Parameters

Dataset A and the table of student numbers; the three components' retention times and the pharmacopoeia's limits and disregard limit; the part of each run searched and the stretch where the noise is measured; how the detection settings follow from the noise and the narrowest peak; how the bounds and the drop-line baseline are found; the plot ranges; then your oil, set in section 2. Leave `SAMPLE` as `CHANGE-ME` until section 2.

In [ ]:
DATA_DIR = Path("data")                # every run, and the table of who has which oil
RUN_A = DATA_DIR / "L13A.cdf"          # Dataset A, section 1's worked example: run A of oil L13
SAMPLES_CSV = DATA_DIR / "samples.csv"  # every student's oil, by student number, section 2
COMPONENTS_MIN = {"camphor": 40.37, "linalool": 42.88, "linalyl_acetate": 43.29}   # retention times, min, by the monograph's elution order
LIMITS_PCT = {"camphor": (None, 1.2), "linalool": (20.0, 45.0), "linalyl_acetate": (25.0, 47.0)}   # Ph. Eur. 1338, area percent; None: no lower limit
DISREGARD_PCT = 0.05         # Ph. Eur. 1338: a peak under 0.05% of the total is left out of the total
RUN_MIN = (5.8, 69.5)        # min, the part of the run searched: the solvent ends before 5.8 min and the shortest run is 70 min
KEEP_MIN = (5.9, 69.4)       # min, the apexes kept, clear of the two edges of that part
QUIET_MIN = (25.0, 33.0)     # min, a stretch with no peak above the noise, where the noise is measured
WIDE_PEAK_MV = 1.0           # mV, the prominence of the peaks whose widths are measured for the narrowest
PROMINENCE_OF_LIMIT = 0.5    # prominence as a fraction of the height of a 0.05% peak at the narrowest width
WIDTH_OF_NARROWEST = 0.5     # width as a fraction of the narrowest width at half height; distance is that whole width
NOISE_SIGMAS = 5             # the lowest prominence section 1 tries, in multiples of the noise
HIGH_PROMINENCE_MV = 0.5     # mV, a prominence read off the plot, the highest section 1 tries
SEARCH_POINTS = 300          # a bound is searched at most this far from its apex: 0.5 min at 10 points per second
ON_BASELINE_POINTS = 180     # drop-line: points each side of a bound, 0.3 min, searched for the lowest signal
ON_BASELINE_MV = 0.05        # drop-line: a bound this close to that lowest signal, mV, is on the baseline
MATCH_MIN = 0.10             # a component is the kept peak within this of its retention time, min
ZOOM_MV = 3.0                # mV, the zoomed plots show 0 to this, where the small peaks are
DETAIL_MIN = (44.0, 52.0)    # min, the stretch of Dataset A where section 1 marks each setting's peaks
SHOW_HALF_MIN = 0.6          # min each side of the least-resolved peak that section 3 draws
SAMPLE = "CHANGE-ME"          # your oil, for example "L19", from data/samples.csv by your student number, section 2

## 1. Explore: which peaks count, and where each one ends

Dataset A is one run of a lavender oil that passes: the solvent near 4.4 min, then the oil's peaks from 6 to 70 min on a baseline near 1 mV. The question for this section is how the detection settings and the integration bounds set each component's area percent. The first cell defines `read_run`, which returns a run's time axis, its signal and the data system's own peak table, then plots the run whole and from 39.5 to 44 min, where camphor sits 2.5 min before the main pair, linalool then linalyl acetate.

In [ ]:
def read_run(path):
    """One GC-FID run: the time axis (min), the signal (mV) and the data system's peak table."""
    f = netcdf_file(path, "r", mmap=False)
    y = f.variables["ordinate_values"].data.astype(float)                  # mV
    dt = float(f.variables["actual_sampling_interval"].data)               # min between points, a single number
    table = pd.DataFrame({"t_min": f.variables["peak_retention_time"].data.astype(float),
                          "area_pct": f.variables["peak_area_percent"].data.astype(float)})
    f.close()
    return np.arange(y.size) * dt, y, table


t_a, y_a, table_a = read_run(RUN_A)
in_run_a = (t_a >= RUN_MIN[0]) & (t_a <= RUN_MIN[1])
print(f"{y_a.size} points, {(t_a[1] - t_a[0]) * 60:.1f} s apart, {t_a[-1]:.0f} min; largest {y_a.max():.0f} mV at {t_a[np.argmax(y_a)]:.2f} min (the solvent)")
print(f"largest after {RUN_MIN[0]} min: {y_a[in_run_a].max():.0f} mV at {t_a[in_run_a][np.argmax(y_a[in_run_a])]:.2f} min; "
      f"the data system's table lists {len(table_a)} peaks")

near_pair = (t_a > 39.5) & (t_a < 44.0)
fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(10, 4))
ax_full.plot(t_a, y_a, color="black", linewidth=0.8)
ax_full.set_xlabel("retention time (min)")
ax_full.set_ylabel("FID signal (mV)")
ax_full.set_title("Dataset A, the whole run")
ax_zoom.plot(t_a[near_pair], y_a[near_pair], color="black", linewidth=0.8)
for name, t_peak in COMPONENTS_MIN.items():
    ax_zoom.axvline(t_peak, color="gray", linestyle=":", linewidth=0.8)
ax_zoom.set_ylim(0, ZOOM_MV)
ax_zoom.set_xlabel("retention time (min)")
ax_zoom.set_ylabel("FID signal (mV)")
ax_zoom.set_title("Dataset A, 0 to 3 mV; dotted lines at the three components")
fig.tight_layout()
plt.show()

At full scale the run is one 1,413 mV solvent peak at 4.39 min, with the oil's largest peak, linalool at 42.89 min, only 124 mV; from 39.5 to 44 min at 0 to 3 mV, camphor is a 1.9 mV peak at 40.37 min on a baseline near 1.0 mV, with a shoulder of about 1.2 mV at 40.53 min, a 1.3 mV peak at 41.77 min, and linalool and linalyl acetate off the scale at 42.88 and 43.29 min.

A peak counts if it can reach the pharmacopoeia's **disregard limit**, 0.05% of the total area; anything smaller is left out of the total. Two numbers turn that limit into settings. The **noise** is the scatter of the signal where no peak is: the standard deviation of the differences between neighboring points from 25 to 33 min, divided by √2 because each difference carries two points' noise. The **narrowest full width at half maximum (FWHM)**, among peaks at least 1 mV prominent, sets how close two real apexes can be and how narrow a real peak is. The next cell measures both.

In [ ]:
def detect(t, y, prominence, width, distance):
    """The peaks find_peaks returns between RUN_MIN, kept where the apex is inside KEEP_MIN: indices into t."""
    found, _ = find_peaks(np.where((t >= RUN_MIN[0]) & (t <= RUN_MIN[1]), y, 0), prominence=prominence, width=width, distance=distance)
    return found[(t[found] > KEEP_MIN[0]) & (t[found] < KEEP_MIN[1])]


quiet_a = (t_a > QUIET_MIN[0]) & (t_a < QUIET_MIN[1])
noise_a = np.std(np.diff(y_a[quiet_a])) / np.sqrt(2)                      # mV
wide_a = detect(t_a, y_a, WIDE_PEAK_MV, None, None)
fwhm_a = peak_widths(np.where(in_run_a, y_a, 0), wide_a, rel_height=0.5)[0]   # points
narrowest_a = fwhm_a.min()                                                # points
points_per_s = 1 / ((t_a[1] - t_a[0]) * 60)
print(f"noise {noise_a:.4f} mV, from {QUIET_MIN[0]:.0f} to {QUIET_MIN[1]:.0f} min")
print(f"{wide_a.size} peaks at least {WIDE_PEAK_MV} mV prominent; FWHM from {narrowest_a:.1f} points ({narrowest_a / points_per_s:.1f} s) "
      f"at {t_a[wide_a[np.argmin(fwhm_a)]]:.2f} min to {fwhm_a.max():.0f} points; median {np.median(fwhm_a):.0f}")

Each peak is integrated between two **bounds**, the lowest points toward its neighbors' apexes, searched at most 300 points away. The **drop-line** rule draws the baseline through the bounds that sit on the true baseline and splits a cluster by vertical lines at its valleys; the **valley-skim** rule, in section 3, draws a straight line under each peak between its own bounds. Area percent is each area over the sum, with peaks under 0.05% dropped and the percent computed again. To set a prominence, the next cell finds the total at 5 times the noise, then the height of a 0.05% peak as narrow as the narrowest. Half that height is the prominence.

In [ ]:
def peak_bounds(y, peaks):
    """Each peak's (left, right) bound: the lowest point toward each neighboring apex, at most SEARCH_POINTS away."""
    bounds = []
    for k, i in enumerate(peaks):
        lo = max(i - SEARCH_POINTS, peaks[k - 1] if k > 0 else 0)
        hi = min(i + SEARCH_POINTS, peaks[k + 1] if k < peaks.size - 1 else y.size - 1)
        bounds.append((lo + int(np.argmin(y[lo:i + 1])), i + int(np.argmin(y[i:hi + 1]))))
    return bounds


def peak_baselines(t, y, peaks, rule):
    """Each peak's bounds and the baseline under it, mV, by the rule "drop" or "valley"."""
    bounds = peak_bounds(y, peaks)
    if rule == "valley":
        return [(lo, hi, np.interp(t[lo:hi + 1], t[[lo, hi]], y[[lo, hi]])) for lo, hi in bounds]
    ends = sorted({i for pair in bounds for i in pair})
    on = [i for i in ends if y[i] - y[max(i - ON_BASELINE_POINTS, 0):i + ON_BASELINE_POINTS + 1].min() <= ON_BASELINE_MV]
    line = np.interp(np.arange(y.size), on, y[on])
    return [(lo, hi, line[lo:hi + 1]) for lo, hi in bounds]


def peak_areas(t, y, peaks, rule):
    """Each peak's area above its baseline, mV min."""
    return np.array([trapezoid(y[lo:hi + 1] - base, t[lo:hi + 1]) for lo, hi, base in peak_baselines(t, y, peaks, rule)])


def area_percent(areas):
    """Area percent of the total, with peaks under DISREGARD_PCT left out (NaN) and the rest computed again."""
    pct = 100 * areas / areas.sum()
    kept = pct >= DISREGARD_PCT
    return np.where(kept, 100 * areas / areas[kept].sum(), np.nan)


def component(times_min, pct, t_peak):
    """The area percent of the kept peak within MATCH_MIN of t_peak; 0 if none is kept there."""
    near = (np.abs(np.asarray(times_min) - t_peak) < MATCH_MIN) & np.isfinite(pct)
    return float(np.asarray(pct)[near].max()) if near.any() else 0.0


low_a = detect(t_a, y_a, NOISE_SIGMAS * noise_a, WIDTH_OF_NARROWEST * narrowest_a, narrowest_a)
total_a = peak_areas(t_a, y_a, low_a, "drop").sum()                        # mV min
gauss = np.sqrt(np.pi / (4 * np.log(2)))                                  # a Gaussian's area over its height times its FWHM
limit_height_a = DISREGARD_PCT / 100 * total_a / (gauss * narrowest_a / points_per_s / 60)   # mV
settings_a = {"prominence": round(PROMINENCE_OF_LIMIT * limit_height_a, 3), "width": round(WIDTH_OF_NARROWEST * narrowest_a, 1),
              "distance": int(round(narrowest_a))}
print(f"at {NOISE_SIGMAS} times the noise: {low_a.size} peaks, total {total_a:.2f} mV min")
print(f"a 0.05% peak at the narrowest FWHM is {limit_height_a:.3f} mV tall")
print(f"settings: prominence {settings_a['prominence']} mV, width {settings_a['width']} points, distance {settings_a['distance']} points")

A lower prominence finds noise, which the disregard limit then drops; a higher one misses real peaks between 0.05 and 0.1%, and every component's share grows as the total shrinks. The next cell detects at three prominences, 5 times the noise, the setting just printed, and 0.5 mV, with the same width and distance, and prints what each finds, keeps and misses, and linalool's area percent. It also detects once with no `distance`, which finds a flat-topped apex twice. The plot marks each setting's peaks from 44 to 52 min.

In [ ]:
tries_a = {f"{NOISE_SIGMAS} x noise": NOISE_SIGMAS * noise_a, "the disregard limit": settings_a["prominence"], "read off the plot": HIGH_PROMINENCE_MV}
found_a, rows = {}, []
for name, prominence in tries_a.items():
    found = detect(t_a, y_a, prominence, settings_a["width"], settings_a["distance"])
    areas = peak_areas(t_a, y_a, found, "drop")
    pct = area_percent(areas)
    found_a[name] = found
    rows.append({"setting": name, "prominence_mV": prominence, "found": found.size, "kept": int(np.isfinite(pct).sum()),
                 "kept_total_mV_min": areas[np.isfinite(pct)].sum(), "linalool_pct": component(t_a[found], pct, COMPONENTS_MIN["linalool"])})
tries = pd.DataFrame(rows)
print(tries.round(3).to_string(index=False))
print(f"the data system's table: {len(table_a)} peaks")

kept_limit = found_a["the disregard limit"][np.isfinite(area_percent(peak_areas(t_a, y_a, found_a["the disregard limit"], "drop")))]
pct_limit = area_percent(peak_areas(t_a, y_a, found_a["the disregard limit"], "drop"))
missed = [i for i in kept_limit if np.abs(t_a[found_a["read off the plot"]] - t_a[i]).min() > settings_a["distance"] / points_per_s / 60]
small = [i for i in missed if component(t_a[found_a["the disregard limit"]], pct_limit, t_a[i]) < 2 * DISREGARD_PCT]
print(f"kept at the disregard limit and not found at {HIGH_PROMINENCE_MV} mV: {len(missed)} peaks, {len(small)} of them between 0.05 and 0.1%")

twin = detect(t_a, y_a, settings_a["prominence"], settings_a["width"], None)
gap = np.argmin(np.diff(t_a[twin]))
print(f"no distance: {twin.size} peaks; the closest two at {t_a[twin[gap]]:.3f} and {t_a[twin[gap + 1]]:.3f} min, {np.diff(t_a[twin])[gap] * 60:.1f} s apart")

detail = (t_a >= DETAIL_MIN[0]) & (t_a <= DETAIL_MIN[1])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_a[detail], y_a[detail], color="black", linewidth=0.8)
for (name, found), marker, lift in zip(found_a.items(), ["x", "o", "^"], [0.15, 0.3, 0.45]):
    shown = found[(t_a[found] >= DETAIL_MIN[0]) & (t_a[found] <= DETAIL_MIN[1])]
    ax.plot(t_a[shown], y_a[shown] + lift, linestyle="none", marker=marker, fillstyle="none", color="black",
            label=f"{name}: {shown.size} here, {found.size} in the run")
ax.set_ylim(0, ZOOM_MV)
ax.set_xlabel("retention time (min)")
ax.set_ylabel("FID signal (mV)")
ax.set_title(f"Dataset A, {DETAIL_MIN[0]:.0f} to {DETAIL_MIN[1]:.0f} min: the peaks each prominence finds")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

From 44 to 52 min, 5 times the noise marks 26 apexes, most of them bumps of 0.1 to 0.2 mV on a baseline near 1.05 mV, the disregard-limit setting marks 13, every peak that rises clearly above the baseline, and 0.5 mV marks 7, missing the six peaks near 44.7, 44.9, 46.6, 47.2, 49.4 and 50.0 min that stand 0.2 to 0.4 mV above the baseline.

The **hand-built table** is the method of this section at the disregard-limit setting and the drop-line rule: the three components' area percent, beside the data system's own `peak_area_percent` for the same peaks. Section 2 checks a skill against this table, so the next cell prints it to two decimals, as the skill will, with the number of peaks each total includes.

In [ ]:
kept_a = int(np.isfinite(pct_limit).sum())
hand_a = pd.DataFrame({"hand_pct": [component(t_a[found_a["the disregard limit"]], pct_limit, t0) for t0 in COMPONENTS_MIN.values()],
                       "data_system_pct": [component(table_a["t_min"], table_a["area_pct"], t0) for t0 in COMPONENTS_MIN.values()]},
                      index=list(COMPONENTS_MIN))
hand_a["difference"] = hand_a["hand_pct"] - hand_a["data_system_pct"]
print(f"Dataset A, prominence {settings_a['prominence']} mV, width {settings_a['width']} points, distance {settings_a['distance']} points, drop-line: "
      f"{found_a['the disregard limit'].size} found, {kept_a} kept; the data system's table {len(table_a)}")
print(hand_a.round(2).to_string())

**Explain.** In two sentences: the hand-built table's linalool reads below the data system's. From the counts and the differences printed, what is the difference from, and does it change Dataset A's call against 20.0 to 45.0%?

*Your answer:*

## 2. Apply: your oil, by hand and by a skill

Your oil is a different commercial product, run three times, A, B and C, by the same method. Its components, its small peaks and its call differ from Dataset A's and from your neighbor's. The question for your oil: does each of its three components sit inside the pharmacopoeia's range, and by how much compared with the spread of the three runs? First you write section 1's method as a skill and check it on Dataset A, then you choose the settings for your runs, run the skill on them, and build the same table by hand.

**A SKILL.md for a repeated step**, today's AI practice, new. A skill is a file of instructions that Claude Code runs as a command when you type its name with `/`, so that a step you repeat is done the same way each time. In VS Code's Explorer, in the repository's top folder (where `README.md` is), make the folder `.claude/skills/peak-report/` and in it a file named `SKILL.md`. Paste the template below into it. Then replace the four blanks with section 1's values: `PROMINENCE`, `WIDTH` and `DISTANCE` with the three settings section 1 printed, and `RULE` with `drop`. Save the file. The repository's `.gitignore` keeps `.claude/skills/` out of git.

```markdown
---
name: peak-report
description: Peak report for one GC-FID run of lavender oil, a .cdf file in this session's data folder. Use when asked for a peak report or for the Ph. Eur. components of a run.
---

# Peak report for one GC-FID run of lavender oil

The run to report is the path in the arguments: $ARGUMENTS

Compute it in one Python script run in this repository's `.venv`. Do not read or edit any notebook.

1. Read the run with `scipy.io.netcdf_file(path, "r", mmap=False)`. The signal is `ordinate_values` in mV; the time step is `actual_sampling_interval` in min, a single number (`float(...data)`), and time starts at 0.
2. Detect peaks with `scipy.signal.find_peaks` on a copy of the signal set to 0 where the time is below 5.8 min or above 69.5 min, with prominence PROMINENCE mV, width WIDTH points and distance DISTANCE points. Keep the peaks whose time is above 5.9 and below 69.4 min.
3. Each peak's left bound is the lowest point from the previous peak's apex (or 300 points before its own apex, whichever is nearer) to its own apex; its right bound is the lowest point from its own apex to the next peak's apex (or 300 points after, whichever is nearer). 300 points is 0.5 min.
4. The integration rule is RULE.
   - valley: the area above the straight line joining the peak's own two bounds.
   - drop: a bound is on the baseline when it lies within 0.05 mV of the lowest signal from 180 points (0.3 min) before it to 180 points after it. The baseline is the straight line joining consecutive on-baseline bounds, and the area is above that baseline between the peak's bounds.
   Integrate with `scipy.integrate.trapezoid` over time in min.
5. Area percent is 100 times each area over the sum of all areas. Drop every peak under 0.05%, the Ph. Eur. disregard limit, and compute the percent again over the peaks kept.
6. Report, under a line that states the parameters and the rule:
   - one table of every peak kept, by time: retention time (min, 3 decimals) and area percent (2 decimals);
   - one table for camphor (40.37 min), linalool (42.88 min) and linalyl acetate (43.29 min), each the kept peak within 0.10 min of that time: area percent, the Ph. Eur. range (camphor at most 1.2%; linalool 20.0 to 45.0%; linalyl acetate 25.0 to 47.0%) and inside or outside;
   - the data system's own `peak_area_percent` for the same three peaks, matched by `peak_retention_time` in the same file.
```

Claude Code reads a new skills folder only when a conversation starts, so in the Claude Code prompt box type `/` and choose **Clear conversation**. Type `/skills` to see `peak-report` listed, then type this, exactly as written, since the command is the skill's name and a file:

`/peak-report sessions/S12-peak-detection/data/L13A.cdf`

Read the skill's three values against section 1's hand-built table. A skill that follows the method agrees with it within 0.01 points, the last digit of a rounding; if a value differs by more, find the step in `SKILL.md` that the assistant read differently, for example the parameter line the report starts with, the rule or a window, change it, and run the command again. Add a markdown cell under this one (hover just below this cell and click **+ Markdown**) holding these lines, and fill the first now:

```
**Prompt record**

- **My prompt:**
- **The skill's Dataset A values against my hand-built ones, and my runs' against TASK 3's:**
- **What I changed in SKILL.md, or what I verified:**
```

Fill the second and third lines after TASK 3. Your prompt record is one candidate for part (b) of the judgment cell.

**[TASK 1]** Find your student number in `data/samples.csv` and set `SAMPLE` in the Parameters cell to the oil beside it, for example `"L19"`. Run the Parameters cell, then the next cell. It reads your three runs, `data/L19A.cdf`, `data/L19B.cdf` and `data/L19C.cdf` for L19, and measures run A's noise, narrowest FWHM and total by section 1's method. Until `SAMPLE` is set, it stops with `FileNotFoundError: [Errno 2] No such file or directory`, naming `CHANGE-MEA.cdf`.

In [ ]:
runs = {rep: read_run(DATA_DIR / f"{SAMPLE}{rep}.cdf") for rep in ("A", "B", "C")}   # FileNotFoundError until SAMPLE is set
for rep, (t, y, table) in runs.items():
    in_run = (t >= RUN_MIN[0]) & (t <= RUN_MIN[1])
    print(f"{SAMPLE}{rep}: {t[-1]:.0f} min, largest after {RUN_MIN[0]} min {y[in_run].max():.0f} mV at {t[in_run][np.argmax(y[in_run])]:.2f} min; "
          f"the data system's table lists {len(table)} peaks")

t_b, y_b, table_b = runs["A"]
noise_b = np.std(np.diff(y_b[(t_b > QUIET_MIN[0]) & (t_b < QUIET_MIN[1])])) / np.sqrt(2)
wide_b = detect(t_b, y_b, WIDE_PEAK_MV, None, None)
fwhm_b = peak_widths(np.where((t_b >= RUN_MIN[0]) & (t_b <= RUN_MIN[1]), y_b, 0), wide_b, rel_height=0.5)[0]
narrowest_b = fwhm_b.min()
total_b = peak_areas(t_b, y_b, detect(t_b, y_b, NOISE_SIGMAS * noise_b, WIDTH_OF_NARROWEST * narrowest_b, narrowest_b), "drop").sum()
limit_height_b = DISREGARD_PCT / 100 * total_b / (gauss * narrowest_b / points_per_s / 60)
print(f"{SAMPLE}A: noise {noise_b:.4f} mV; narrowest FWHM {narrowest_b:.1f} points at {t_b[wide_b[np.argmin(fwhm_b)]]:.2f} min; "
      f"total {total_b:.2f} mV min; a 0.05% peak at that width {limit_height_b:.3f} mV tall")
print(f"Dataset A: noise {noise_a:.4f} mV; narrowest FWHM {narrowest_a:.1f} points; total {total_a:.2f} mV min; {limit_height_a:.3f} mV")

**[TASK 2]** Choose the settings for your runs from the printout above, by section 1's rule: the prominence, in mV, is half the height of a 0.05% peak; the width, in points, is half the narrowest FWHM; the distance, in points, is the narrowest FWHM, a whole number. Round as section 1 did. The cell then prints what each run finds and keeps with your settings and with section 1's; a count that differs is a peak your settings and Dataset A's treat differently. Until it is filled, the cell stops with `TypeError: float() argument must be a string or a real number, not 'ellipsis'`.

In [ ]:
prominence_mV = float(...)   # [TASK 2] half the height of a 0.05% peak in your run A, mV, 3 decimals
width_points = float(...)    # [TASK 2] half your run A's narrowest FWHM, points, 1 decimal
distance_points = int(...)   # [TASK 2] your run A's narrowest FWHM, points, a whole number
settings_b = {"prominence": prominence_mV, "width": width_points, "distance": distance_points}
for rep, (t, y, table) in runs.items():
    counts = []
    for settings in (settings_b, settings_a):
        found = detect(t, y, settings["prominence"], settings["width"], settings["distance"])
        counts.append((found.size, int(np.isfinite(area_percent(peak_areas(t, y, found, "drop"))).sum())))
    print(f"{SAMPLE}{rep}: yours {counts[0][0]} found, {counts[0][1]} kept; section 1's {counts[1][0]} found, {counts[1][1]} kept")

Now the skill on your runs. In `SKILL.md`, change `PROMINENCE`, `WIDTH` and `DISTANCE` to your TASK 2 values and save; an edit to a skill that already exists needs no new conversation. Then type, with your oil in place of L19:

`/peak-report sessions/S12-peak-detection/data/L19A.cdf`

and the same for runs B and C. If a report's first line still states section 1's settings, choose **Clear conversation** as before and type the command again. Keep the three tables in the chat for the comparison after TASK 3.

**[TASK 3]** Build your hand-built table yourself, in a code cell you add under this one (**+ Code**), with section 1's functions `detect`, `peak_areas`, `area_percent` and `component`, your TASK 2 settings and the drop-line rule, as section 1's last cell did for Dataset A. It sets one name, `hand`: a pandas DataFrame with one row per run, indexed `A`, `B` and `C`, and the columns `camphor`, `linalool` and `linalyl_acetate`, each the area percent. The cell after it prints `hand` beside the data system's values, the mean and standard deviation of the three runs, and each component's call, and stops with `NameError: name 'hand' is not defined` until your cell sets it.

In [ ]:
def nearest_limit(name, value):
    """The limit of LIMITS_PCT[name] nearest to value, and value's distance from it in percentage points."""
    limits = [x for x in LIMITS_PCT[name] if x is not None]
    nearest = min(limits, key=lambda x: abs(value - x))
    return nearest, abs(value - nearest)


system = pd.DataFrame({rep: {name: component(table["t_min"], table["area_pct"], t_peak) for name, t_peak in COMPONENTS_MIN.items()}
                       for rep, (t, y, table) in runs.items()}).T
print(f"{SAMPLE}, your settings, drop-line: area percent by hand, then by the data system")
print(pd.concat({"hand": hand, "data system": system}, axis=1).round(2).to_string())   # NameError until your cell sets hand
rows = []
for name in COMPONENTS_MIN:
    low, high = LIMITS_PCT[name]
    mean = hand[name].mean()
    nearest, margin = nearest_limit(name, mean)
    rows.append({"component": name, "mean_pct": mean, "sd_pct": hand[name].std(ddof=1), "range_pct": f"{low or 0} to {high}",
                 "nearest_limit_pct": nearest, "margin_points": margin, "margin_in_sd": margin / hand[name].std(ddof=1),
                 "runs_inside": int(sum((low is None or v >= low) and v <= high for v in hand[name]))})
calls_b = pd.DataFrame(rows).set_index("component")
print(calls_b.round(2).to_string())

**Explain.** Before you compare: in two sentences, give your oil's call, each component inside or outside its range, and for the component nearest a limit, its margin against the spread of your three runs. Then compare with a neighbor: set both oils' calls and margins side by side, and say whose call a second integration rule could change. Add one sentence on where your two answers differ and what in your data explains it.

*Your answer:*

## 3. Analyze: drop-line or valley skim

Two integration rules through the same peaks and the same bounds: treatment A, the drop-line, which section 1 and TASK 3 used, and treatment B, the valley skim, a straight line under each peak between its own two bounds. They agree wherever a valley returns to the baseline and differ where two peaks overlap, since the skim cuts the shared foot from both. Note that the skim's line can cross a flank: where one bound lies out on the flat baseline and the other at a high valley, the line runs above the signal on the peak's outer flank, and the skim subtracts the area between them there. The next cell computes your three runs both ways, prints each component's mean under each rule, the difference, and the margin to the nearest limit, then draws run A around its least-resolved valley, the peak whose area the skim cuts most, with each rule's baseline.

In [ ]:
def show_bounds(ax, t, y, peaks, rule, window):
    """The trace within window (min) and, under each peak inside it, the baseline the rule integrates above."""
    style = {"drop": dict(color="gray", linestyle="--"), "valley": dict(color="black", linestyle=":")}[rule]
    inside = [(lo, hi, base) for lo, hi, base in peak_baselines(t, y, peaks, rule) if t[lo] >= window[0] and t[hi] <= window[1]]
    shown = (t >= window[0]) & (t <= window[1])
    ax.plot(t[shown], y[shown], color="black", linewidth=0.8)
    for k, (lo, hi, base) in enumerate(inside):
        ax.plot(t[lo:hi + 1], base, linewidth=1.2, label=f"{rule} baseline" if k == 0 else None, **style)


by_rule = {}
for rule in ("drop", "valley"):
    rows = {}
    for rep, (t, y, table) in runs.items():
        found = detect(t, y, settings_b["prominence"], settings_b["width"], settings_b["distance"])
        pct = area_percent(peak_areas(t, y, found, rule))
        rows[rep] = {name: component(t[found], pct, t_peak) for name, t_peak in COMPONENTS_MIN.items()}
    by_rule[rule] = pd.DataFrame(rows).T
rows = []
for name in COMPONENTS_MIN:
    low, high = LIMITS_PCT[name]
    row = {"component": name}
    for rule, label in (("drop", "A_drop"), ("valley", "B_valley")):
        row[f"{label}_mean_pct"] = by_rule[rule][name].mean()
        row[f"{label}_runs_inside"] = int(sum((low is None or v >= low) and v <= high for v in by_rule[rule][name]))
    row["B_minus_A_points"] = row["B_valley_mean_pct"] - row["A_drop_mean_pct"]
    row["nearest_limit_pct"], row["A_margin_points"] = nearest_limit(name, row["A_drop_mean_pct"])
    row["A_sd_pct"] = by_rule["drop"][name].std(ddof=1)
    rows.append(row)
rules_b = pd.DataFrame(rows).set_index("component")
print(f"{SAMPLE}, three runs, your settings")
print(rules_b.round(2).to_string())

found_b = detect(t_b, y_b, settings_b["prominence"], settings_b["width"], settings_b["distance"])
areas_drop_b = peak_areas(t_b, y_b, found_b, "drop")
cut_b = np.where(np.isfinite(area_percent(areas_drop_b)), areas_drop_b - peak_areas(t_b, y_b, found_b, "valley"), -np.inf)   # mV min, kept peaks
k = int(np.argmax(cut_b))
worst = found_b[k]
print(f"run A: the rules differ most at the peak at {t_b[worst]:.2f} min, {areas_drop_b[k]:.3f} mV min by drop-line and {areas_drop_b[k] - cut_b[k]:.3f} by valley skim")

window = (t_b[worst] - SHOW_HALF_MIN, t_b[worst] + SHOW_HALF_MIN)
fig, ax = plt.subplots(figsize=(8, 4))
show_bounds(ax, t_b, y_b, found_b, "drop", window)
show_bounds(ax, t_b, y_b, found_b, "valley", window)
ax.set_xlabel("retention time (min)")
ax.set_ylabel("FID signal (mV)")
ax.set_title(f"{SAMPLE}A around its least-resolved valley, {t_b[worst]:.2f} min")
ax.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**Explain.** In two sentences, say which rule the drawn valley justifies and why, from whether the signal between the two peaks returns to the baseline, and whether your oil's call depends on the choice, citing the largest difference between the rules against the margin to the nearest limit. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): as the analyst, does your oil meet the Ph. Eur. profile for camphor, linalool and linalyl acetate? Cite each component's mean and the spread of your three runs, the nearest limit and your margin to it, how far the other integration rule moves it, and the data system's value, and say what a wrong call would cost: an oil cut with lavandin or synthetic linalool sold as pharmacopoeial, or a genuine lot rejected. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your data. Your prompt record from section 2 is the natural source; the reason has to cite a value from your output, such as the skill's Dataset A value against your hand-built one, not a general rule. This session's AI practice, named in the notes, is *a SKILL.md for a repeated step*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"SciPy      {scipy.__version__}")
print(f"Matplotlib {matplotlib.__version__}")